# Genenerating cuboids from segmentatin annotations

In [1]:
# if needed, install
!pip install open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.7/399.7 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.8 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: werkzeug
    Found existing installation: Werkzeug 3.1.2
    Uninstalling Werkzeug-3.1.2:
      Successfully uninstalled Werkzeug-3.1.2
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7

## Define functions

In [2]:
import numpy as np
import open3d as o3d
import openpyxl
import math
import json
import copy

### Load and save data

In [3]:
# load point cloud from file
def load_point_cloud_from_file(file_path):
    pcd = o3d.io.read_point_cloud(file_path)
    return np.asarray(pcd.points)

def save_annotations_to_file(annotations_cuboids, file_path):
    # Open the file to write in
    with open(file_path, "w") as file:
        # Write each string to a new line
        for string in annotations_cuboids:
            file.write(string + "\n")

    print("The annotations were successfully saved to " + file_path)

def load_annotation_from_file(file_path):
    with open(file_path) as f:
     annotation = json.load(f)
    return annotation

### Generation of the cuboids

In [41]:
def cuboid3D_segmentsai_annotation (minVector, maxVector):
    position=[(minVector[0]+maxVector[0])/2,(minVector[1]+maxVector+[1])/2, (minVector[2]+maxVector[2])/2 ]
    dimension = [abs(maxVector[0]-minVector[0]), abs(maxVector[1]-minVector[1]), abs(maxVector[2]-minVector[2])]
    rotation = [0,0,0]
    return [position,rotation,dimension]

def cuboid3D_mmdetection3d_annotation_segmentsai (minVector, maxVector, category_name):
    centerBox=[(minVector[0]+maxVector[0])/2,(minVector[1]+maxVector[1])/2, (minVector[2]+maxVector[2])/2 ]
    dimensionBox= [abs(maxVector[0]-minVector[0]), abs(maxVector[1]-minVector[1]), abs(maxVector[2]-minVector[2])]
    headingAngle = 0
    return str(round(centerBox[0],2))+" "+str(round(centerBox[1],2))+" "+str(round(centerBox[2],2))+" "+str(round(dimensionBox[0],2))+" "+str(round(dimensionBox[1],2))+" "+str(round(dimensionBox[2],2))+" "+str(headingAngle)+" "+category_name


def minVector(points):
    return np.min(points, axis=0)

def maxVector (points):
    return np.max(points, axis=0)


def sample_plants_parts_list (sheet_path,  worksheet_name, sample_name, sample_name_column, plant_parts_column):
    # return list of lists
    # each list represent one plant and content object ids of plant parts
    # source: worksheet with plant parts records
    wb_Annotation = openpyxl.load_workbook(sheet_path)
    worksheet = wb_Annotation[worksheet_name]
    sample_plants_parts_list=[]
    for i in range(1,worksheet.max_row):
     if worksheet.cell(i,sample_name_column).value == sample_name:
       plant_items_list=[]
       for j in range(plant_parts_column,plant_parts_column+15):
         if worksheet.cell(i,j).value != None:
           plant_items_list.append(worksheet.cell(i,j).value)
       sample_plants_parts_list.append(plant_items_list)
    return sample_plants_parts_list

def plant_points (sample_points, sample_name, annotation_json, plant_parts_list):
    # return sample points represent plant
    # source sample points, name, json file contain label info, plant parts list
    plant_points=[]
    js_temp = json.dumps(annotation_json)
    js_release = json.loads(js_temp)
    for item in js_release['dataset']['samples']:
      if item['name'] == sample_name:
       sample_label=item
       if  sample_label['labels']['ground-truth']!=None:
         plant_point_index=[]
         object_points= sample_label['labels']['ground-truth']['attributes']['point_annotations']
         sample_objects= sample_label['labels']['ground-truth']['attributes']['annotations']
         index=-1
         for point in sample_label['labels']['ground-truth']['attributes']['point_annotations']:
          index=index+1
          if int(point) in plant_parts_list:
             plant_point_index.append(index)
         if (len(plant_point_index)>0):
            plantpoints=np.asarray(sample_points)
            plant_points=plantpoints[np.asarray(plant_point_index)]
    return plant_points









## Parameters

In [5]:
# connect google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [33]:
# Main working directory
working_path = '/content/drive/Shared drives/3D point clouds/Data/Scientific data repo/'

point_clouds_path = working_path + "Data/Point clouds/"
point_cloud_name = "50-27-0_1-20220601T033109.pcd"

annotation_worksheet_path = working_path + "Data/Annotation data.xlsx"
sheet_name = "Annotated files"
annotations_path = working_path + "Data/Segments-ai annotations.json"
output_path = working_path + "Code/Cuboids generation/cuboids.txt"

## Execution

In [44]:
# load the point cloud file
pcd_file = load_point_cloud_from_file(point_clouds_path+point_cloud_name)
file_name = point_cloud_name.split(".")[0]

#load annotation using the segments.ai format
annotations = load_annotation_from_file(annotations_path)

# load the individual plant records (segmentation object IDs)
s_plants_parts_list= sample_plants_parts_list(annotation_worksheet_path,sheet_name,file_name, 8,9 )

annotations_cuboids = []
# generate the cuboid annotations, to list
for plant_parts_list in s_plants_parts_list:
  s_plant_points=(plant_points(pcd_file, point_cloud_name, annotations, plant_parts_list))
  if (len(s_plant_points)>0):
    annotation = cuboid3D_segmentsai_annotation(minVector(s_plant_points),maxVector(s_plant_points))
    annotation = cuboid3D_mmdetection3d_annotation_segmentsai(minVector(s_plant_points),maxVector(s_plant_points),"Plant")
    annotations_cuboids.append(annotation)
    print(annotation)
  else:
    print("No plant points found")

# save the annotations to file
save_annotations_to_file(annotations_cuboids, output_path)


159.3 72.53 -80.32 112.94 138.64 87.77 0 Plant
51.84 49.13 -88.46 78.93 129.01 79.53 0 Plant
-93.96 -91.35 -74.02 112.39 71.09 69.49 0 Plant
82.06 -67.2 -71.56 58.47 112.77 65.36 0 Plant
-76.19 59.92 -84.8 62.86 74.91 72.44 0 Plant
The annotations were successfully saved to /content/drive/Shared drives/3D point clouds/Data/Scientific data repo/Code/Cuboids generation/cuboids.txt
